# 通达信备用输入可行性核验

## tl;dr

当前登记存续池5,224股，原始文件存在5,224份；原输入记录缺失2,045条，复权构建通过5,138股。整池门槛未通过；未覆盖生产数据，也未重算收益。

## Context & Methods

只研究原输入的5,224只存续代码；日期2016-08-22至2026-09-30。逐文件验证32字节记录、日期主键、量为股/额为人民币元、正OHLC和非负有限量额。沿用登记的比例前复权纯函数，特殊类别拒绝而不猜补。

### Key Assumptions

当前除权资料不能证明历史公告可得性；跨源差异不自动证明哪一方正确。这里只判断是否具备建立同池备用输入的基础。

## Data

原面板与TDX .day/gbbq路径、SHA256、协议和逐股结果见registration.json、protocol.json、inventory.csv。以下从冻结CSV独立重算汇总并校验来源哈希。

In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd
root = Path('F:\\88888.投资\\08.选股\\a_-share_-rps_-system')
out = root / 'output/tdx_input_20261007/v1'
summary = json.loads((out/'summary.json').read_text(encoding='utf-8'))
inventory = pd.read_csv(out/'inventory.csv')
receipt = json.loads((out/'final_manifest.json').read_text(encoding='utf-8'))
for relative, expected in receipt['files'].items():
    assert hashlib.sha256((root/relative).read_bytes()).hexdigest() == expected
assert len(inventory) == 5224 and inventory.ts_code.is_unique
assert inventory.missing_original_keys.sum() == summary['missing_original_keys']
assert inventory.adjustment_status.eq('valid').sum() == summary['adjustment_valid_codes']
eligible = inventory.raw_status.eq('valid') & inventory.adjustment_status.eq('valid') & inventory.missing_original_keys.eq(0)
assert bool(eligible.all()) == summary['gate_passed']
print('冻结文件与整池门槛重算一致')

冻结文件与整池门槛重算一致


## Results

门槛逐项列出，避免把未验收代码删除后称作整池通过。

In [2]:
pd.DataFrame([
    ['登记代码数', len(inventory)], ['原始文件存在', inventory.file_exists.sum()],
    ['原始验证通过', inventory.raw_status.eq('valid').sum()],
    ['缺失原股票日期', inventory.missing_original_keys.sum()],
    ['复权构建通过', inventory.adjustment_status.eq('valid').sum()],
    ['同池覆盖与复权均通过', eligible.sum()]
], columns=['检查项','数量'])

,检查项,数量
0,登记代码数,5224
1,原始文件存在,5224
2,原始验证通过,5224
3,缺失原股票日期,2045
4,复权构建通过,5138
5,同池覆盖与复权均通过,5137


In [3]:
inventory.loc[~eligible, ['ts_code','raw_status','missing_original_keys','adjustment_status','reason']].head(8)

,ts_code,raw_status,missing_original_keys,adjustment_status,reason
55,000155.SZ,valid,0,rejected,unsupported adjustment category
72,000410.SZ,valid,0,rejected,unsupported adjustment category
85,000430.SZ,valid,0,rejected,unsupported adjustment category
102,000520.SZ,valid,0,rejected,unsupported adjustment category
104,000523.SZ,valid,0,rejected,unsupported adjustment category
156,000595.SZ,valid,0,rejected,unsupported adjustment category
173,000620.SZ,valid,0,rejected,unsupported adjustment category
191,000656.SZ,valid,0,rejected,unsupported adjustment category


## Takeaways

原输入与本地来源哈希前后不变。若整池门槛未通过，须先处理资料/复权缺口，不能偷偷缩小股票池、拿原供应商价格混合TDX量额后称为纯TDX，或更改策略规则救收益。若通过，也须建立单独日期版本、重算RPS并复核调整价，再按原协议复核收益；不等于实盘资格。